# This notebook generates CMG dat files for running CMG simulations

## Step 1: Set up a base CMG model
* Prepare a base CMG dat file and add it to data/dat_file_templates
* Note CMG requires initializing stress state using a reference block. For the JD_geothermal grid (139, 248, 23), reservoir is between k=5 and 20. Use block (50, 248, 5) as reference block for *STRESSGRAD calculation. Its grid top = 2365.807 m and bottom = 2383.534 m.

## Step 2: Sample uncertain parameters

* Note PORO/PERMX pairs are NOT sampled more than once (from file names instead of files)

### Option 1: Monte Carlo sampling

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from parameter_sampling import latin_hypercube_sampling

################## User Inputs ############################## 
# Note: 1) stress gradients are effective ones (required by CMG) after subtracting 10; 
#       2) stress gradients are negative due to CMG DIR DOWN convention
param_bounds = {
    # 'temp_grad_C/m'             : [0.025,      0.035],
    'pres_init_kPa'             : [24000,      25000],
    'c_water_/kPa'              : [4.7e-7,     4.8e-7],
    'c_rock_/kPa'               : [3e-7,       5e-7],
    'FTM'                       : [0.0,        0.5],
    'hcap_reservoir_J/m3-C'     : [2.304e6,    3.117e6],
    'thcond_reservoir_J/m-day-C': [2.387e5, 2.920e5],
    'E_kPa'                     : [30e6,       35e6],
    'PR'                        : [0.2,        0.4],
    'SH_MPa/km'                 : [-10 * 1.1,  -10 * 0.9],
    'Sh_MPa/km'                 : [-4.5 * 1.1, -4.5 * 0.9],
    'Sv_MPa/km'                 : [-14 * 1.1,  -14 * 0.9],
    'SH_azi_deg'                : [0,          20],
}    # param_name : [lower, upper]
sampling_results = latin_hypercube_sampling(
    name_prefix = '260720_test', # prefix for the output file name
    random_seed = 11, # random seed for Latin Hypercube sampling
    property_file_names_path = repo_root/'results'/'Vienna_geothermal'/'property_file_names'/'test_geothermal_filenames.csv', # path for the property file names
    output_file_path = repo_root/'results'/'Vienna_geothermal'/'sim_dat_files', # path for the output file
    n_samples = 2,  # number of unique poro/permx pairs to sample (should be <= number of available pairs)
    param_names   = list(param_bounds.keys()),
    lower_bounds  = [b[0] for b in param_bounds.values()],
    upper_bounds  = [b[1] for b in param_bounds.values()],
    property_list = ['PORO','PERMX','TEMP'], # list of properties (CMG keywords) to sample from
    ref_block_top_depth = 2365.807,   # initial stress reference block, for the JD_Sula_2005_gmc grid, the reference block is (50, 1, 6) 
    ref_block_bottom_depth = 2383.534,
    show_results = True
)
################## End of User Inputs #######################

,temp_grad_C/m,pres_init_kPa,c_water_/kPa,c_rock_/kPa,FTM,hcap_reservoir_J/m3-C,thcond_reservoir_J/m-day-C,E_kPa,PR,SH_MPa/km,...,beta,cos_2beta,sin_2beta,sigma_x_grad,sigma_y_grad,tau_xy_grad,sigma_x_ref,sigma_y_ref,sigma_z_ref,tau_xy_ref
0,0.034357,24750.4,4.719930e-07,3.971310e-07,0.463018,2333180.0,263473.0,34675600.0,0.205167,-10.62190,...,-86.6284,-0.993083,-0.117419,-4.68665,-10.6013,-0.349663,11129.2,25174.6,34945.5,830.334
1,0.028623,24431.0,4.760600e-07,4.329640e-07,0.121904,2785000.0,277367.0,30047700.0,0.379549,-9.55373,...,-75.9160,-0.881566,-0.472060,-4.58066,-9.2407,-1.247680,10877.6,21943.6,31095.3,2962.820


### Option 2: Monte Carlo sampling + importance sampling

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from parameter_sampling import latin_hypercube_and_importance_sampling

# Note: 1) stress gradients are effective ones (required by CMG) after subtracting 10; 
#       2) stress gradients are negative due to CMG DIR DOWN convention
# OMV_values = [20e6, 0.3, x, 14.6, 22.7, 300]
# base_values = [20e6, 0.3, 28, 16.5, 22.7, 310]
sampling_results = latin_hypercube_and_importance_sampling(
    ################## User Inputs ############################## 
    name_prefix = 'test_251209', # prefix for the output file name
    random_seed = 15, # random seed for Latin Hypercube sampling
    property_file_names_path = repo_root/'results'/'property_file_names'/'property_file_names_seed7.csv', # path for the property file names
    output_file_path = repo_root/'results'/'sim_files', # path for the output file
    n_samples = 83,  # number of unique poro/permx pairs to sample (should be <= number of available pairs)
    param_names = ['E_GPa', 'PR', 'SH_MPa/km', 'Sh_MPa/km', 'Sv_MPa/km', 'SH_azi_deg', 'A_m2'],
    lower_bounds = [15e6, 0.2, -18 * 1.1, -6.5 * 1.1, -12.7 * 1.1, 300, 16985344.51*0.9],
    upper_bounds = [25e6, 0.4, -18 * 0.9, -6.5 * 0.9, -12.7 * 0.9, 320, 16985344.51*1.1],
    ref_block_top_depth = 670.7188,   # initial stress reference block, for the JD_Sula_2005_gmc grid, the reference block is (50, 1, 6) 
    ref_block_bottom_depth = 671.9521,
    show_results = True,
    # below is for importance sampling for the Sula CCS
    alpha = 0.9,
    beta = 0.9,
    proposal_SH_azi_low = 319,
    proposal_SH_low = 18*1.05,
    show_summary = True
    ################## End of User Inputs #######################
)

## Step 3: generate CMG dat files based on the sampled parameters

In [3]:
import pandas as pd
from pathlib import Path
import sys

repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from generate_dat_files import generate_dat_files

# df_params = pd.read_csv(base_path/'results'/'sim_files'/f"{sampling_results['name_prefix']}_sampled_params_seed{sampling_results['random_seed']}.csv")
generate_dat_files(
    ################## User Inputs ############################## 
    df_parameters = sampling_results['param_dataframe'],
    template_file_path = repo_root/'data'/'Vienna_geothermal'/'dat_file_templates'/'JD_geothermal_base_model_v3.dat',
    save_folder_path = repo_root/'results'/'Vienna_geothermal'/'sim_dat_files'/f"{sampling_results['name_prefix']}_dat_files"
    ################## End of User Inputs #######################
)

Generated 2 dat files successfully.
